In [ ]:
import sys, json, os, time
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

p = "../data/processed/"
fail_dates = pd.read_csv(p + "fail_dates.csv", parse_dates=["fail_date"])
failed_set = set(fail_dates["serial_number"])

d = np.load(p + "oof_scores.npz")
df = pd.DataFrame({"s": d["serial"], "y": d["y"], "gru": d["gru"], "ae": d["ae"]})
healthy = df[~df["s"].isin(failed_set)].groupby("s")[["gru", "ae"]].max()
pos = df[df["y"] == 1].groupby("s")[["gru", "ae"]].max()

# a level = a score that only this share of healthy drives ever reach in ~5 months
LEVELS = {"medium": 0.97, "high": 0.99, "critical": 0.995}
thresholds = {"risk": {k: float(np.quantile(healthy["gru"], q)) for k, q in LEVELS.items()},
              "anomaly": float(np.quantile(healthy["ae"], 0.99))}

evaluation = {"protocol": "5-fold CV grouped by drive, 189 evaluable failed drives",
              "risk": {k: {"healthy_drives_flagged": float((healthy["gru"] >= thresholds["risk"][k]).mean()),
                           "failed_drives_detected": float((pos["gru"] >= thresholds["risk"][k]).mean())}
                       for k in LEVELS},
              "anomaly": {"healthy_drives_flagged": float((healthy["ae"] >= thresholds["anomaly"]).mean()),
                          "failed_drives_detected": float((pos["ae"] >= thresholds["anomaly"]).mean())}}
print(json.dumps({"thresholds": thresholds, "evaluation": evaluation}, indent=2))

In [ ]:
from dp_ml.preprocess import make_windows_for_drive, to_log_space, WINDOW, HORIZON, MAX_FILL

data = pd.read_csv(p + "st12000nm0008_2026q1q2_clean.csv.gz", parse_dates=["date"])
FEATURE_COLS = [c for c in data.columns if c.startswith("smart_")]
fail_date_lookup = dict(zip(fail_dates["serial_number"], fail_dates["fail_date"]))

Xs, ys, ss = [], [], []
for serial, df_drive in data.groupby("serial_number"):
    w, l = make_windows_for_drive(df_drive, fail_date_lookup.get(serial), FEATURE_COLS, stride=3)
    if w:
        Xs.append(np.stack(w)); ys.append(np.array(l, dtype=np.float32)); ss += [serial] * len(l)

X = to_log_space(np.concatenate(Xs)); del Xs
y = np.concatenate(ys)
serial = np.array(ss)
F = X.shape[-1]
healthy_mask = ~np.isin(serial, list(failed_set))
print("windows:", X.shape, "| positives:", int(y.sum()))

In [ ]:
from dp_ml.models import RiskNet, LSTMAE

SEED = 42
torch.manual_seed(SEED)
rk = np.random.default_rng(SEED)

def norm_stats(idx, n):
    s = X[rk.choice(idx, size=min(n, len(idx)), replace=False)]
    m = s.reshape(-1, F).mean(axis=0)
    sd = s.reshape(-1, F).std(axis=0)
    sd[sd == 0] = 1.0
    return m, sd

pos_idx = np.where(y == 1)[0]
neg_idx = np.where(y == 0)[0]
mean, std = norm_stats(np.arange(len(y)), 200_000)

gru = RiskNet(F, mean, std)
opt = torch.optim.Adam(gru.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(3.0))
for ep in range(10):
    gru.train()
    neg = rk.choice(neg_idx, size=len(pos_idx) * 30, replace=False)
    order = rk.permutation(np.concatenate([pos_idx, neg]))
    total = 0.0
    for i in range(0, len(order), 256):
        b = order[i:i + 256]
        opt.zero_grad()
        loss = loss_fn(gru(torch.from_numpy(X[b])), torch.from_numpy(y[b]))
        loss.backward()
        nn.utils.clip_grad_norm_(gru.parameters(), 1.0)
        opt.step()
        total += loss.item() * len(b)
    print(f"GRU epoch {ep + 1} | loss {total / len(order):.4f}")

In [ ]:
healthy_idx = np.where(healthy_mask)[0]
mean_h, std_h = norm_stats(healthy_idx, 100_000)

ae = LSTMAE(F, mean_h, std_h)
opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
for ep in range(8):
    ae.train()
    order = rk.choice(healthy_idx, size=150_000, replace=False)
    total = 0.0
    for i in range(0, len(order), 512):
        b = order[i:i + 512]
        opt.zero_grad()
        loss = ae.score(torch.from_numpy(X[b])).mean()
        loss.backward()
        nn.utils.clip_grad_norm_(ae.parameters(), 1.0)
        opt.step()
        total += loss.item() * len(b)
    print(f"AE epoch {ep + 1} | loss {total / len(order):.4f}")

In [ ]:
os.makedirs("../models", exist_ok=True)
torch.save(gru.state_dict(), "../models/risk_gru.pt")
torch.save(ae.state_dict(), "../models/anomaly_lstmae.pt")

config = {
    "version": "gru-ae-v1",
    "drive_model": "ST12000NM0008",
    "training_data": "Backblaze Drive Stats 2026 Q1+Q2",
    "feature_cols": FEATURE_COLS,          # model input = these + gap_filled (last)
    "window": WINDOW, "horizon_days": HORIZON, "max_fill": MAX_FILL,
    "preprocessing": "forward-fill gaps up to max_fill days, add gap_filled flag, log1p; "
                     "z-score and clamp to +-10 happen inside the model",
    "thresholds": thresholds,
    "evaluation": evaluation,
}
with open("../models/config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)
print(os.listdir("../models"))

In [ ]:
from dp_ml.inference import Predictor

gru.eval(); ae.eval()
pred = Predictor("../models")
counts = data.groupby("serial_number").size()

# a failed drive, with its history cut 3 days before the failure
for s, fd in fail_date_lookup.items():
    if counts.get(s, 0) < 40:
        continue
    hist = data[(data["serial_number"] == s) & (data["date"] <= fd - pd.Timedelta(days=3))]
    out = pred.predict(hist)
    if out is not None:
        print("FAILED drive:", out)
        break

# a healthy drive, plus a check that training and serving give identical scores
hs = next(s for s in counts.index if s not in failed_set and counts[s] >= 150)
dfh = data[data["serial_number"] == hs]
out = pred.predict(dfh)
print("HEALTHY drive:", out)

w, _ = make_windows_for_drive(dfh, None, FEATURE_COLS, stride=1)
x = torch.from_numpy(to_log_space(w[-1])[None])
with torch.no_grad():
    r = float(torch.sigmoid(gru(x))[0]); a = float(ae.score(x)[0])
print("train/serve match:", abs(r - out["risk_score"]) < 1e-5, abs(a - out["anomaly_score"]) < 1e-5)